[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C02_Post_Training_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与 logprob 冒烟测试

**目标**：确认 `posttrain` 环境可用，并跑通全课最核心的原语——**序列对数概率（logprob）**。

本 notebook 纯 CPU、秒级运行，做四件事：

1. 依赖版本自检（torch / transformers / peft / trl / datasets / numpy / scipy）
2. 计算设备与 API key 检测（key 全部可选，缺了不影响核心模块）
3. **logprob 冒烟测试**：用一个 3-token 词表的玩具"语言模型"算逐 token logprob —— SFT 最大化它、PPO/GRPO 用它做 importance ratio、DPO 用它做隐式奖励、KL 惩罚是两组它的差
4. ✏️ 两道必做练习：`sequence_logprob` 与 `kl_divergence`（assert 自动判分）

> 运行前确认右上角 kernel 是 **Post-Training Course**（`posttrain`）。配置方法见 `00_overview.html` 第 4 节。

In [ ]:
# ===== 环境自检：依赖版本 / 计算设备 / API key =====
import importlib
import os
import sys

RED, GREEN, YELLOW, RESET = "\033[31m", "\033[32m", "\033[33m", "\033[0m"
print(f"Python {sys.version.split()[0]}  ({sys.executable})\n")

# --- 1) 依赖版本（缺的标红并给出安装命令） ---
packages = ["torch", "transformers", "peft", "trl", "datasets", "numpy", "scipy"]
missing = []
for name in packages:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "?")
        print(f"{GREEN}✓{RESET} {name:<14} {ver}")
    except ImportError:
        missing.append(name)
        print(f"{RED}✗ {name:<14} 未安装  →  pip install {name}{RESET}")

if missing:
    print(f"\n{RED}缺 {len(missing)} 个依赖，一次性安装：pip install " + " ".join(missing) + RESET)
else:
    print(f"\n{GREEN}依赖齐全 ✅{RESET}")

# --- 2) 计算设备（本课 CPU 即可，GPU/MPS 只加速可选环节） ---
print()
if "torch" not in missing:
    import torch
    if torch.cuda.is_available():
        device = "cuda"
        print(f"device = cuda ({torch.cuda.get_device_name(0)})")
    elif torch.backends.mps.is_available():
        device = "mps"
        print("device = mps (Apple Silicon)")
    else:
        device = "cpu"
        print("device = cpu（本课全部核心模块 CPU 可跑，无需担心）")
else:
    device = None
    print(f"{RED}torch 未安装，请先安装再继续{RESET}")

# --- 3) API key（布尔检测，不打印内容；全部可选） ---
print()
for key, usage in [("HF_TOKEN", "拉取需授权的 HF 模型/数据集（可选）"),
                   ("OPENAI_API_KEY", "模块 06 LLM-as-judge 对比实验（可选）")]:
    is_set = bool(os.environ.get(key))
    mark = f"{GREEN}已设置{RESET}" if is_set else f"{YELLOW}未设置{RESET}"
    print(f"{key:<16} {mark}  —— {usage}")

## logprob 冒烟测试：全课最核心的原语

一个自回归语言模型在每个位置 $t$ 输出一行 logits $z_t \in \mathbb{R}^{|V|}$，经 softmax 得到下一个 token 的分布。
给定实际出现的 token 序列 $y = (y_1, \dots, y_T)$，**逐 token logprob** 与**序列总 logprob** 为：

$$\log \pi(y_t \mid y_{<t}) = \log\mathrm{softmax}(z_t)[y_t], \qquad \log \pi(y) = \sum_{t=1}^{T} \log \pi(y_t \mid y_{<t})$$

为什么它是"原子操作"：

| 算法 | 怎么用 logprob |
|---|---|
| SFT（模块 01） | 最大化示范序列的 $\log\pi_\theta(y\mid x)$（只对 response 位置） |
| PPO（模块 03） | importance ratio $= \exp(\log\pi_\theta - \log\pi_{\text{old}})$ |
| DPO（模块 04） | 隐式奖励 $= \beta(\log\pi_\theta - \log\pi_{\text{ref}})$，chosen 减 rejected |
| GRPO（模块 05） | 同 PPO 的 ratio + 序列级 KL 估计 |
| KL 惩罚（贯穿） | $\log\pi_\theta - \log\pi_{\text{ref}}$ 的期望 |

下面手搭一个 **3-token 词表的玩具"语言模型"**：没有任何神经网络，就是一张固定的 logits 矩阵（每行 = 模型在该位置的输出）。
规模小到每个数都能手算验证——这正是本课"玩具实现"层的风格。

> 数值要点：永远用 `log_softmax`（内部做 log-sum-exp 稳定化），不要 `softmax` 之后再 `log`。

In [ ]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)

# 玩具词表与"冻结的语言模型"：每行是模型在该步输出的 logits
vocab = ["A", "B", "C"]                      # |V| = 3
logits = torch.tensor([[2.0, 0.0, -1.0],     # 第 1 步：偏爱 "A"
                       [0.0, 1.5,  0.0],     # 第 2 步：偏爱 "B"
                       [1.0, 1.0,  1.0],     # 第 3 步：均匀分布
                       [0.0, 0.0,  3.0]])    # 第 4 步：强烈偏爱 "C"

token_ids = torch.tensor([0, 1, 2, 2])       # 实际序列 "A B C C"

logprobs = F.log_softmax(logits, dim=-1)             # (T, V)：每行和为 1 的分布取 log
tok_lp = logprobs[torch.arange(len(token_ids)), token_ids]   # 按位置取实际 token 的 logprob

print(f"{'step':<6}{'token':<8}{'logprob':>10}{'prob':>9}")
for t, (i, lp) in enumerate(zip(token_ids, tok_lp)):
    print(f"{t:<6}{vocab[i]:<8}{lp.item():>10.4f}{lp.exp().item():>9.4f}")

total_lp = tok_lp.sum()
print(f"\n序列总 logprob = {total_lp.item():.4f}")
print(f"序列概率       = {total_lp.exp().item():.6f}")
print(f"困惑度 PPL     = {(-total_lp / len(token_ids)).exp().item():.4f}   (= exp(平均负 logprob))")

# 冒烟断言：每行 log_softmax 取 exp 后应当和为 1
assert torch.allclose(logprobs.exp().sum(dim=-1), torch.ones(4))
print("\n✅ logprob 冒烟测试通过 —— 后训练的核心原语已就绪")

## ✏️ 练习 1：实现 `sequence_logprob(logits, token_ids)`

把上面的冒烟测试封装成函数——后面每个模块都会用到它（或它的变体）。

**任务**：给定 logits 矩阵（形状 `(T, V)`）和实际 token 序列 `token_ids`（形状 `(T,)`），返回**序列总 logprob**（0 维 tensor）。

**提示**（3 行以内可完成）：
1. `F.log_softmax(logits, dim=-1)` 得到 `(T, V)` 的逐位置 log 分布；
2. 用 `torch.arange(T)` 配合 `token_ids` 做花式索引，取出每个位置实际 token 的 logprob；
3. `.sum()` 求和。

自测用例里有一组 **2×3 logits 可手算**：两行 logits 各自都是常数行（`[0,0,0]` 与 `[5,5,5]`），
softmax 对整行平移不变，所以每行都是均匀分布，每个 token 的 logprob 都是 $-\ln 3$，总 logprob $= -2\ln 3 \approx -2.1972$。

In [ ]:
import torch
import torch.nn.functional as F

def sequence_logprob(logits: torch.Tensor, token_ids: torch.Tensor) -> torch.Tensor:
    '''计算序列总 logprob。

    Args:
        logits:    (T, V) 每个位置的未归一化分数
        token_ids: (T,)   实际出现的 token id
    Returns:
        0 维 tensor：sum_t log_softmax(logits[t])[token_ids[t]]
    '''
    # TODO: 1) log_softmax 得到逐位置 log 分布
    # TODO: 2) 按位置取出实际 token 的 logprob（花式索引）
    # TODO: 3) 求和返回
    raise NotImplementedError

In [ ]:
# ===== 练习 1 自测（全过则通过） =====
import math

# 用例 A（手算）：两行常数 logits → 均匀分布 → 总 logprob = -2 ln 3
logits_a = torch.tensor([[0.0, 0.0, 0.0],
                         [5.0, 5.0, 5.0]])
out_a = sequence_logprob(logits_a, torch.tensor([2, 0]))
assert out_a.dim() == 0, "应返回 0 维 tensor（标量）"
assert torch.isclose(out_a, torch.tensor(-2 * math.log(3.0)), atol=1e-6), f"期望 {-2*math.log(3.0):.4f}, 得到 {out_a.item():.4f}"

# 用例 B（边界：单 token 序列）
out_b = sequence_logprob(torch.tensor([[0.0, 0.0, 0.0]]), torch.tensor([1]))
assert torch.isclose(out_b, torch.tensor(-math.log(3.0)), atol=1e-6)

# 用例 C（数值稳定性：大 logits 不应溢出）
out_c = sequence_logprob(torch.tensor([[100.0, 0.0, 0.0]]), torch.tensor([0]))
assert out_c.item() <= 0 and out_c.item() > -1e-6, "选中概率≈1 的 token，logprob 应≈0"
out_d = sequence_logprob(torch.tensor([[100.0, 0.0, 0.0]]), torch.tensor([1]))
assert out_d.item() < -99, "选中概率≈0 的 token，logprob 应≈-100"
assert torch.isfinite(out_d), "大 logits 下不应出现 inf/nan（用 log_softmax）"

# 用例 D（与冒烟测试一致）
assert torch.isclose(sequence_logprob(logits, token_ids), total_lp, atol=1e-6)

print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `kl_divergence(logp, logq)`

KL 散度是 RLHF 的"缰绳"：$\max_\pi \mathbb{E}[r] - \beta\,\mathrm{KL}(\pi\|\pi_{\text{ref}})$ 里那个 KL，
也是模块 03/05 里 reward shaping 与策略漂移监控的核心量。离散分布下：

$$\mathrm{KL}(P \,\|\, Q) = \sum_i p_i \log \frac{p_i}{q_i} = \sum_i e^{\log p_i}\,(\log p_i - \log q_i)$$

**任务**：输入两个**log 概率向量** `logp`、`logq`（同形状 1 维 tensor，各自 `exp` 后和为 1），返回 $\mathrm{KL}(P\|Q)$（0 维 tensor）。

**提示**（2 行以内可完成）：`(logp.exp() * (logp - logq)).sum()`。注意约定 LLM 场景下输入是 logprob 而非概率——直接在 log 域做差最稳定。

自测会检查：与 `scipy.stats.entropy(p, q)` 一致（`entropy` 双参数形式算的就是 KL，自然对数底）、$\mathrm{KL}(P\|P)=0$、非负、**不对称**。

In [ ]:
import torch

def kl_divergence(logp: torch.Tensor, logq: torch.Tensor) -> torch.Tensor:
    '''离散分布 KL(P || Q)。

    Args:
        logp, logq: (N,) 两个分布的 log 概率（exp 后各自和为 1）
    Returns:
        0 维 tensor：sum_i exp(logp_i) * (logp_i - logq_i)
    '''
    # TODO: 在 log 域做差，再用 p 加权求和
    raise NotImplementedError

In [ ]:
# ===== 练习 2 自测（全过则通过） =====
from scipy.stats import entropy

p = torch.tensor([0.5, 0.3, 0.2], dtype=torch.float64)
q = torch.tensor([0.2, 0.5, 0.3], dtype=torch.float64)
logp, logq = p.log(), q.log()

# 与 scipy 对照（entropy(p, q) 即 KL(P||Q)，自然对数底）
kl_pq = kl_divergence(logp, logq)
assert kl_pq.dim() == 0, "应返回 0 维 tensor"
assert torch.isclose(kl_pq, torch.tensor(entropy(p.numpy(), q.numpy())), atol=1e-10), \
    f"与 scipy.stats.entropy(p, q) 不一致: {kl_pq.item():.6f} vs {entropy(p.numpy(), q.numpy()):.6f}"

# KL(P||P) = 0（精确为 0：log 域差恰好是零向量）
assert kl_divergence(logp, logp).item() == 0.0, "KL(P, P) 应为 0"

# 非负 + 不对称
kl_qp = kl_divergence(logq, logp)
assert kl_pq.item() > 0 and kl_qp.item() > 0
assert not torch.isclose(kl_pq, kl_qp), "KL 散度不对称：KL(P||Q) ≠ KL(Q||P)"

print("✅ 练习 2 通过")
print(f"   KL(P||Q) = {kl_pq.item():.6f}   KL(Q||P) = {kl_qp.item():.6f}（不对称）")

## 📖 参考答案

先自己做，再对照。两题合计不到 10 行——但它们会在后面 7 个模块里以各种变体反复出现。

In [ ]:
# ===== 练习 1 参考答案（先自己做，再对照） =====
def sequence_logprob(logits: torch.Tensor, token_ids: torch.Tensor) -> torch.Tensor:
    logprobs = F.log_softmax(logits, dim=-1)              # (T, V)
    tok_lp = logprobs[torch.arange(logits.shape[0]), token_ids]   # (T,) 按位置取实际 token
    return tok_lp.sum()

# 重跑上面的自测 cell 验证。
# 真实 LLM 场景的两点差异（模块 01 详讲）：
#   1) logits 要先错位：位置 t 的 logits 预测的是 token t+1（shift）；
#   2) 配合 mask 只对 response 位置求和 —— 这就是 loss masking。

In [ ]:
# ===== 练习 2 参考答案（先自己做，再对照） =====
def kl_divergence(logp: torch.Tensor, logq: torch.Tensor) -> torch.Tensor:
    return (logp.exp() * (logp - logq)).sum()

# 重跑上面的自测 cell 验证。
# 工程注记：PPO/GRPO 训练中无法对整个序列空间求和，实际用采样估计 KL，
# 常见的低方差估计量 k3 = (r - 1) - log r, r = exp(logq - logp)，模块 05 会实现。

## 每模块算力需求一览

全部**核心内容 CPU 可跑**；"真实模型环节"均为可选，带 mock 回退，跳过不影响主线。

| 模块 | 核心玩具实现（CPU，必跑） | 真实模型环节（可选） |
|---|---|---|
| 00 环境 | logprob / KL，秒级 | — |
| 01 SFT | loss masking 手写交叉熵，分钟级 | Qwen2.5-0.5B-Instruct + LoRA（下载约 1 GB；CPU 约 20 分钟 / GPU 数分钟） |
| 02 奖励模型 | Bradley-Terry 玩具 RM + overoptimization 模拟，分钟级 | Qwen2.5-0.5B + reward head |
| 03 RLHF/PPO | 纯 PyTorch 手写 PPO 循环，CPU 数分钟 | — |
| 04 DPO 家族 | DPO/IPO/SimPO 玩具对比，分钟级 | trl `DPOTrainer` + Qwen2.5-0.5B |
| 05 RLVR/GRPO | 手写 GRPO + format reward，CPU 数分钟 | — |
| 06 对齐评测 | sycophancy / length bias 统计模拟，分钟级 | `OPENAI_API_KEY` 时的 LLM-as-judge 对比 |
| 07 前沿 | weak-to-strong 玩具复现，CPU 数分钟 | — |

## ✅ 环境就绪，下一步

两道练习都打出 ✅ 后，环境与核心原语就绪。进入 **[模块 01 · SFT 与 loss masking](../01_sft/01_讲解.html)**：
先读 `01_讲解.html`（chat template、masked 交叉熵推导、LoRA），再做配套 notebook——
你会发现练习 1 的 `sequence_logprob` 加上一个 mask，就是 SFT 的损失函数。